In [5]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
import pandas as pd

# Inisialisasi SparkSession
spark = SparkSession.builder.appName("Tugas11_NPM_NamaLengkap").getOrCreate()

# Membaca dataset
df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
df.show(5)

26/11/18 19:33:18 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows



A. Feature Engineering 

Gunakan StringIndexer untuk mengubah kolom nama_menu, cabang, dan waktu_pesan menjadi indeks numerik. Gabungkan seluruh fitur (qty, harga_satuan, dan ketiga kolom hasil indexing) menjadi satu kolom features menggunakan VectorAssembler.

In [6]:
# StringIndexer untuk kolom kategorikal
indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx")
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx")

# Menerapkan indexer secara berurutan
df_indexed = indexer_menu.fit(df).transform(df)
df_indexed = indexer_cabang.fit(df_indexed).transform(df_indexed)
df_indexed = indexer_waktu.fit(df_indexed).transform(df_indexed)

# Menggabungkan fitur dengan VectorAssembler
assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "menu_idx", "cabang_idx", "waktu_idx"],
    outputCol="features"
)
df_model = assembler.transform(df_indexed)
df_model.select("features", "total_penjualan").show(5, truncate=False)

+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
+-------------------------+------------------+
only showing top 5 rows



B. Train/Test Split & Pelatihan Model 

Bagi data menjadi 75% training dan 25% testing (seed=101). Latih model LinearRegression dengan labelCol="total_penjualan".

In [7]:
# Membagi data (75% training, 25% testing) dengan seed=101
train_data, test_data = df_model.randomSplit([0.75, 0.25], seed=101)

# Inisialisasi dan pelatihan model LinearRegression
lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
lr_model = lr.fit(train_data)

print("Model berhasil dilatih.")

26/11/18 19:33:38 WARN Instrumentation: [c1294054] regParam is zero, which might cause numerical instability and overfitting.
26/11/18 19:33:39 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/11/18 19:33:39 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


Model berhasil dilatih.


C. Evaluasi Model 

Hitung dan tampilkan RMSE dan R² dari model pada data testing. Tulis 1-2 kalimat interpretasi: apakah model ini cukup baik untuk dipakai memprediksi omzet?

In [8]:
# Melakukan prediksi pada data testing
predictions = lr_model.transform(test_data)

# Evaluasi RMSE
evaluator_rmse = RegressionEvaluator(predictionCol="prediction", labelCol="total_penjualan", metricName="rmse")
rmse = evaluator_rmse.evaluate(predictions)

# Evaluasi R2
evaluator_r2 = RegressionEvaluator(predictionCol="prediction", labelCol="total_penjualan", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE: {rmse}")
print(f"R-squared (R2): {r2}")

RMSE: 10900.803581280325
R-squared (R2): 0.910679109181288


D. Eksperimen Perbandingan Fitur

Latih model kedua yang HANYA menggunakan fitur qty dan harga_satuan (tanpa informasi menu/cabang/waktu). Bandingkan R² model kedua ini dengan model utama pada bagian C. Tulis kesimpulan: apakah menambahkan informasi menu/cabang/waktu benar-benar meningkatkan akurasi prediksi secara berarti?

In [9]:
# Membuat VectorAssembler baru hanya dengan qty dan harga_satuan
assembler_sederhana = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_sederhana"
)
df_model_sederhana = assembler_sederhana.transform(df_indexed)

# Membagi data kembali dengan seed yang sama
train_data_sed, test_data_sed = df_model_sederhana.randomSplit([0.75, 0.25], seed=101)

# Melatih model kedua
lr_sederhana = LinearRegression(featuresCol="features_sederhana", labelCol="total_penjualan")
lr_model_sederhana = lr_sederhana.fit(train_data_sed)

# Prediksi dan evaluasi model kedua
predictions_sederhana = lr_model_sederhana.transform(test_data_sed)
r2_sederhana = evaluator_r2.evaluate(predictions_sederhana)

print(f"R-squared Model Utama (5 fitur): {r2}")
print(f"R-squared Model Kedua (2 fitur - qty & harga): {r2_sederhana}")

26/11/18 19:34:08 WARN Instrumentation: [b768a2f2] regParam is zero, which might cause numerical instability and overfitting.


R-squared Model Utama (5 fitur): 0.910679109181288
R-squared Model Kedua (2 fitur - qty & harga): 0.9107572632054388


E. Prediksi Skenario Baru (bobot 15%)

Buat sebuah DataFrame baru berisi 3 skenario pesanan (data dummy) (boleh bebas kombinasi menu/cabang/waktu/qty/harga, namun harus melalui tahapan StringIndexer dan VectorAssembler yang sama seperti data training — perhatikan: gunakan model StringIndexer yang sama yang sudah di-fit() sebelumnya, jangan membuat StringIndexer baru). Tampilkan hasil prediksi total_penjualan untuk ketiga skenario tersebut.

In [10]:
# 1. Membuat data skenario baru menggunakan pandas
data_skenario = pd.DataFrame({
    "nama_menu": ["Kopi Susu", "Roti Bakar", "Es Teh"],
    "cabang": ["Semarang", "Yogyakarta", "Magelang"],
    "waktu_pesan": ["Sore", "Pagi", "Malam"],
    "qty": [10, 5, 20],
    "harga_satuan": [18000, 25000, 15000]
})

# Mengubah ke Spark DataFrame
df_skenario = spark.createDataFrame(data_skenario)

# 2. Menerapkan StringIndexer yang SUDAH DI-FIT sebelumnya
# HARUS menggunakan model .fit() yang sudah ada, jangan membuat StringIndexer baru
indexer_menu_model = indexer_menu.fit(df)
indexer_cabang_model = indexer_cabang.fit(df)
indexer_waktu_model = indexer_waktu.fit(df)

df_sken_indexed = indexer_menu_model.transform(df_skenario)
df_sken_indexed = indexer_cabang_model.transform(df_sken_indexed)
df_sken_indexed = indexer_waktu_model.transform(df_sken_indexed)

# 3. Menerapkan VectorAssembler yang sama dari Bagian A
df_sken_features = assembler.transform(df_sken_indexed)

# 4. Melakukan prediksi dengan model utama (lr_model)
hasil_prediksi_skenario = lr_model.transform(df_sken_features)

# Menampilkan hasil
print("=== Hasil Prediksi Skenario Baru ===")
hasil_prediksi_skenario.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction").show()

=== Hasil Prediksi Skenario Baru ===
+----------+----------+-----------+---+------------+------------------+
| nama_menu|    cabang|waktu_pesan|qty|harga_satuan|        prediction|
+----------+----------+-----------+---+------------+------------------+
| Kopi Susu|  Semarang|       Sore| 10|       18000|209430.30185150733|
|Roti Bakar|Yogyakarta|       Pagi|  5|       25000|118689.58789495718|
|    Es Teh|  Magelang|      Malam| 20|       15000| 419343.8968054298|
+----------+----------+-----------+---+------------+------------------+

